# HyperData ↔ Lumen Integration Tutorial

This notebook demonstrates the full workflow:
1. **Pull** a shared dataset from remote HyperData storage
2. **Load** it via Lumen's dataset adapters for training
3. **Augment** — append new samples to the dataset and push back
4. **Train** a model (self-supervised or segmentation)
5. **Push** trained weights back to HyperData with versioning
6. **Pull** weights into a fresh model

## 0. Setup

```bash
uv pip install -e ".[hyperdata]"
```

The cell below sets default environment variables for the shared HyperData
deployment. Override them before running if your server differs.

In [ ]:
import os

# ---- Default HyperData remote configuration ----
# Override any of these before running if your deployment differs.
os.environ.setdefault("HYPERDATA_ENDPOINT", "http://118.180.19.234:8021")
os.environ.setdefault("MINIO_ENDPOINT", "118.180.19.234")
os.environ.setdefault("MINIO_PORT", "9010")
os.environ.setdefault("MINIO_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("MINIO_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_ENDPOINT", "118.180.19.234")
os.environ.setdefault("S3_PORT", "9010")
os.environ.setdefault("S3_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("S3_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_BUCKET", "hyperdata-data")

import numpy as np
import torch
import torch.nn as nn
from hyperdata import HyperData

from lumen.data.hyperdata import (
    HyperDataImageDataset,
    HyperDataSegmentationDataset,
    WeightManager,
    HYPERDATA_AVAILABLE,
)

print(f"HyperData available: {HYPERDATA_AVAILABLE}")
print(f"PyTorch version: {torch.__version__}")
print(f"Remote endpoint: {os.environ['HYPERDATA_ENDPOINT']}")
print(f"S3 endpoint: {os.environ['MINIO_ENDPOINT']}:{os.environ['MINIO_PORT']}")

# Remote S3 URL where the demo dataset is stored
REMOTE_S3_URL = "s3://hyperdata-data/lumen/microscopy-demo.zarr"

## 1. Pull Dataset from Remote HyperData

A shared microscopy demo dataset has been pushed to remote S3 storage.
Pull it to a local directory — HyperData only downloads the chunks you
actually need (IceChunk delta sync).

In [ ]:
DATA_DIR = "/tmp/lumen_hyperdata_demo"

# Pull the shared dataset from remote S3
ds = HyperData(DATA_DIR)
ds.add_remote("origin", REMOTE_S3_URL)
result = ds.pull("origin")
print(f"Pull: success={result.success}, bytes={result.bytes_downloaded}")

# Reopen to see the pulled arrays
ds = HyperData(DATA_DIR)
print(f"\nDataset keys: {ds.keys()}")
print(f"Images shape: {ds['images'].shape}")   # (100, 128, 128) float32
print(f"Masks  shape: {ds['masks'].shape}")     # (100, 128, 128) int64
print(f"Labels shape: {ds['labels'].shape}")    # (100,) int64

## 2. Load Data with Lumen Adapters

### 2a. Self-supervised: image-only dataset

In [ ]:
# Load directly from the HyperData object
img_dataset = HyperDataImageDataset(
    ds,
    array_name="images",
    channels=1,       # keep grayscale
    image_size=64,    # resize to 64×64
)

print(f"Dataset length: {len(img_dataset)}")
sample = img_dataset[0]
print(f"Sample keys:    {list(sample.keys())}")
print(f"Image shape:    {sample['image'].shape}")
print(f"Image dtype:    {sample['image'].dtype}")
print(f"Value range:    [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")

### 2b. Segmentation: paired images + masks

In [ ]:
seg_dataset = HyperDataSegmentationDataset(
    ds,
    image_array="images",
    mask_array="masks",
    image_size=64,
    channels=1,
)

sample = seg_dataset[0]
print(f"Image shape: {sample['image'].shape}")  # (1, 64, 64)
print(f"Mask  shape: {sample['mask'].shape}")    # (64, 64)
print(f"Mask  dtype: {sample['mask'].dtype}")     # torch.int64
print(f"Unique mask values: {sample['mask'].unique().tolist()}")

### 2c. Use with PyTorch DataLoader

In [ ]:
loader = torch.utils.data.DataLoader(img_dataset, batch_size=16, shuffle=True)
batch = next(iter(loader))
print(f"Batch image shape: {batch['image'].shape}")  # (16, 1, 64, 64)

## 3. Augment Dataset and Push Back

Add new synthetic samples to the dataset, then push the updated version
back to remote storage. HyperData's delta sync only uploads the changed
chunks, making this efficient even for large datasets.

In [ ]:
# Generate 20 additional samples (e.g. newly acquired microscopy images)
np.random.seed(99)
new_images = np.random.rand(20, 128, 128).astype(np.float32) * 0.8
new_masks  = np.random.randint(0, 4, size=(20, 128, 128)).astype(np.int64)
new_labels = np.random.randint(0, 3, size=(20,)).astype(np.int64)

# Append to the existing dataset inside a new transaction
old_n = ds["images"].shape[0]
with ds.transaction("append 20 new microscopy images"):
    ds["images"] = np.concatenate([np.array(ds["images"]), new_images])
    ds["masks"]  = np.concatenate([np.array(ds["masks"]),  new_masks])
    ds["labels"] = np.concatenate([np.array(ds["labels"]), new_labels])

print(f"Dataset grew: {old_n} → {ds['images'].shape[0]} images")

# Push the updated dataset to Hub (delta sync + register in catalogue)
# push_to_hub:
#   1. pushes Zarr data to S3 (only changed chunks)
#   2. calls the Hub API to register/update the dataset so it shows in `hd overview`
ds.push_to_hub("@devin/microscopy-demo")
print("push_to_hub completed — dataset registered in HyperData catalogue")

# Reload Lumen adapters to pick up the new data
img_dataset = HyperDataImageDataset(ds, array_name="images", channels=1, image_size=64)
seg_dataset = HyperDataSegmentationDataset(
    ds, image_array="images", mask_array="masks", image_size=64, channels=1,
)
print(f"Updated dataset length: {len(img_dataset)}")

## 4. Train a Simple Model

We'll train a tiny CNN on the augmented dataset to show the end-to-end
flow. In practice you'd use Lumen's `MAETrainer`, `SegmentationTrainer`, etc.

In [ ]:
class TinyCNN(nn.Module):
    """Minimal CNN for demonstration."""
    def __init__(self, in_ch: int = 1, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 8, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(8, num_classes)

    def forward(self, x):
        f = self.features(x).flatten(1)
        return self.head(f)


model = TinyCNN(in_ch=1, num_classes=4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Quick training loop (3 epochs)
loader = torch.utils.data.DataLoader(img_dataset, batch_size=16, shuffle=True)

for epoch in range(3):
    total_loss = 0.0
    for batch in loader:
        imgs = batch["image"]
        # Dummy target for demo (real workflow uses masks/labels)
        targets = torch.randint(0, 4, (imgs.shape[0],))
        optimizer.zero_grad()
        loss = criterion(model(imgs), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss = {total_loss / len(loader):.4f}")

## 5. Push Weights to HyperData

In [ ]:
WEIGHTS_DIR = "/tmp/lumen_hyperdata_demo_weights"

wm = WeightManager(WEIGHTS_DIR)

# Push just the model weights
wm.push_weights(
    model,
    message="TinyCNN after 3 epochs",
    tag="v1.0",
    metrics={"final_loss": total_loss / len(loader)},
)

# Push a full checkpoint (model + optimizer + epoch)
wm.push_checkpoint(
    model, optimizer, epoch=3,
    message="Full checkpoint at epoch 3",
)

print(f"Tags: {wm.list_tags()}")

## 6. Pull Weights into a Fresh Model

In [ ]:
# Create a fresh model with random weights
fresh_model = TinyCNN(in_ch=1, num_classes=4)

# Pull the tagged weights
wm2 = WeightManager(WEIGHTS_DIR)
meta = wm2.pull_weights(fresh_model, tag="v1.0")
print(f"Metadata: {meta}")

# Verify weights match the trained model
for (n1, p1), (n2, p2) in zip(model.named_parameters(), fresh_model.named_parameters()):
    assert torch.allclose(p1, p2), f"Mismatch in {n1}"
print("\nAll weights match — round-trip successful!")

## 7. Resume Training from Checkpoint

In [ ]:
# Restore full checkpoint
resume_model = TinyCNN(in_ch=1, num_classes=4)
resume_opt   = torch.optim.Adam(resume_model.parameters(), lr=1e-3)

ckpt = wm2.pull_checkpoint(resume_model, resume_opt)
print(f"Resumed from epoch {ckpt['epoch']}")
print(f"Checkpoint keys: {list(ckpt.keys())}")

## 8. Loading from a Path (No Existing HyperData Object)

You can also pass a file-system path directly — the adapter will open the
dataset automatically.

In [ ]:
# Pass a path string instead of a HyperData object
dataset_from_path = HyperDataImageDataset(DATA_DIR, array_name="images")
print(f"Loaded {len(dataset_from_path)} images from path")
print(f"Sample shape: {dataset_from_path[0]['image'].shape}")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.rmtree(WEIGHTS_DIR, ignore_errors=True)
print("Cleaned up temp directories.")